# Edge AI Pipeline - Raspberry Pi 4
Notebook này định nghĩa một cấu trúc Pipeline liền mạch, chia làm 5 Tầng để tối ưu hóa hiệu năng.


In [ ]:
import cv2
import time
import numpy as np
import ai_edge_litert.interpreter as tflite
import matplotlib.pyplot as plt
from IPython.display import clear_output


## Tầng 3: Định nghĩa hàm Cắt ROI (Region of Interest)
Việc thu hẹp không gian quan sát (Ví dụ: Chỉ lấy khung cửa) giúp tiết kiệm đáng kể CPU và loại bỏ các nhiễu ở ngoài rìa camera.


In [ ]:
def cat_roi(frame, y_start=100, y_end=400, x_start=150, x_end=450):
    """
    Cắt ảnh theo tọa độ Y và X chỉ định.
    """
    return frame[y_start:y_end, x_start:x_end]


## Tầng 1 & 2: Trừ khung hình và Lọc nhiễu
- **Tầng 1**: Dùng `cv2.absdiff` và `cv2.threshold` (Ngưỡng 25) để tìm điểm ảnh khác biệt (> 500 pixel).
- **Tầng 2**: Lọc qua 3 lớp: Lọc trung vị (`cv2.medianBlur` 3x3) trị nhiễu hạt tiêu, Lọc hình thái học (`cv2.morphologyEx` `MORPH_OPEN`) xóa râu ria, Lọc theo diện tích Contours (>= 40px). Có thêm hàm kiểm tra ảnh mờ (Laplacian variance < 100).


In [ ]:
def kiem_tra_anh_mo(frame_color, nguong_mo=100):
    """
    Kiểm tra xem ảnh có bị mờ không dựa trên phương sai Laplacian.
    """
    gray = cv2.cvtColor(frame_color, cv2.COLOR_BGR2GRAY)
    do_sac_net = cv2.Laplacian(gray, cv2.CV_64F).var()
    return do_sac_net < nguong_mo

def phat_hien_chuyen_dong_co_loc_nhieu(frame_sau_gray, frame_truoc_gray, threshold_val=25, min_contour_area=40, debug=False):
    """
    Kết hợp Tầng 1 (Trừ khung hình) và Tầng 2 (Lọc nhiễu 3 lớp).
    Trả về tổng diện tích (px) của các vùng chuyển động đạt chuẩn.
    """
    # TẦNG 1: Trừ khung hình
    diff = cv2.absdiff(frame_sau_gray, frame_truoc_gray)
    _, threshold_img = cv2.threshold(diff, threshold_val, 255, cv2.THRESH_BINARY)
    raw_pixels = np.sum(threshold_img == 255)
    
    # Nếu Tầng 1 phát hiện dưới 500 pixel thay đổi thô, coi như không có gì (tùy chọn tối ưu cực nhanh)
    if raw_pixels < 500:
        return 0
        
    # TẦNG 2: 3 lớp lọc nhiễu
    # Lớp 1: Lọc trung vị (3x3)
    step1_img = cv2.medianBlur(threshold_img, 3)
    
    # Lớp 2: Lọc hình thái học (Opening)
    kernel_3x3 = np.ones((3, 3), np.uint8)
    step2_img = cv2.morphologyEx(step1_img, cv2.MORPH_OPEN, kernel_3x3)
    
    # Lớp 3: Lọc diện tích Contours
    contours, _ = cv2.findContours(step2_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    total_clean_area = 0
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area >= min_contour_area:
            total_clean_area += area
            
    return total_clean_area


## Tầng 4: Máy trạng thái (State Machine)
Yêu cầu có đủ N khung hình liên tiếp có chuyển động thì mới kích hoạt (Trả về True). Đứt 1 nhịp sẽ reset bộ đếm về 0. Đồng thời quản lý thời gian Cooldown.


In [ ]:
class MayTrangThai:
    def __init__(self, so_khung_lien_tiep=5, thoi_gian_cooldown=5.0):
        self.so_khung_lien_tiep = so_khung_lien_tiep
        self.bo_dem_chuyen_dong = 0
        self.thoi_gian_cooldown = thoi_gian_cooldown
        self.thoi_diem_phat_hien_cuoi = 0.0

    def cap_nhat(self, co_chuyen_dong):
        """
        Cập nhật trạng thái và trả về True nếu cần kích hoạt Tầng 5 (AI).
        """
        # Kiểm tra trạng thái nghỉ (Cooldown)
        if time.time() - self.thoi_diem_phat_hien_cuoi < self.thoi_gian_cooldown:
            return False
            
        if co_chuyen_dong:
            self.bo_dem_chuyen_dong += 1
        else:
            self.bo_dem_chuyen_dong = 0 # Đứt quãng 1 nhịp -> reset
            
        if self.bo_dem_chuyen_dong >= self.so_khung_lien_tiep:
            self.bo_dem_chuyen_dong = 0
            self.thoi_diem_phat_hien_cuoi = time.time() # Bắt đầu Cooldown
            return True
            
        return False


## Tầng 5: Mô Hình AI (TFLite Inference)
Sử dụng `ai_edge_litert.interpreter`. Tự động nhận biết kiểu dữ liệu Quantized (INT8) và bóc tách ma trận siêu tốc mà không dùng vòng For.


In [ ]:
class Edge_TFLite_Model:
    def __init__(self, model_path="yolov8n_int8.tflite", conf_thres=0.5):
        self.conf_thres = conf_thres
        print(f"[Tầng 5] Đang khởi tạo model {model_path}...")
        
        self.interpreter = tflite.Interpreter(model_path=model_path, num_threads=4)
        self.interpreter.allocate_tensors()
        
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()
        
        # Tự động lấy cấu hình đầu vào
        self.input_shape = self.input_details[0]['shape']
        self.height = self.input_shape[1]
        self.width = self.input_shape[2]
        
        # Tự động kiểm tra Quantization
        self.is_quantized = self.input_details[0]['dtype'] == np.int8
        
        if self.is_quantized:
            self.scale, self.zero_point = self.output_details[0]['quantization']
            
        print(f"[Tầng 5] Đã tải model. Shape: {self.width}x{self.height} | INT8: {self.is_quantized}")
        
    def nhan_dien(self, frame_color):
        """
        Chạy TFLite Interpreter trên frame. Trả về True nếu max_score > conf_thres.
        """
        # 1. Tiền xử lý
        img = cv2.resize(frame_color, (self.width, self.height))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.is_quantized:
            input_data = np.expand_dims(img, axis=0)
        else:
            input_data = np.expand_dims(img, axis=0).astype(np.float32) / 255.0
            
        # 2. Suy luận
        self.interpreter.set_tensor(self.input_details[0]['index'], input_data)
        self.interpreter.invoke()
        
        # 3. Hậu xử lý cực nhanh (bóc tách điểm số từ ma trận YOLOv8 [1, 84, 8400])
        output_data = self.interpreter.get_tensor(self.output_details[0]['index'])
        max_conf_raw = np.max(output_data[0][4:, :]) # Điểm số bắt đầu từ hàng thứ 4
        
        # Giải mã Quantization (Dequantize) nếu model là INT8
        if self.is_quantized:
            max_conf = (max_conf_raw - self.zero_point) * self.scale
        else:
            max_conf = max_conf_raw
            
        if max_conf > self.conf_thres:
            return True, max_conf
            
        return False, max_conf


## Chạy Main Pipeline
Kết hợp toàn bộ Tầng 1 -> Tầng 5 thành một luồng thực thi liên tục.


In [ ]:
# Cấu hình chung
FPS = 10
NGUONG_CHUYEN_DONG = 60 # Tối thiểu 60px thay đổi diện tích sau khi đã lọc
MAX_FRAMES_TEST = 100

print("=== BẮT ĐẦU KHỞI TẠO HỆ THỐNG ===")
may_trang_thai = MayTrangThai(so_khung_lien_tiep=5, thoi_gian_cooldown=5.0)
# Khởi tạo mô hình (Comment lại nếu bạn chưa tải file yolov8n_int8.tflite vào thư mục)
try:
    ai_model = Edge_TFLite_Model("yolov8n_int8.tflite", conf_thres=0.5)
    has_model = True
except Exception as e:
    print(f"⚠️ Không khởi tạo được mô hình TFLite ({e}). Sẽ chạy Pipeline bỏ qua Tầng 5.")
    has_model = False

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

ret, frame_truoc_color = cap.read()
if ret:
    # Tầng 3: Cắt ROI NGAY LẬP TỨC để giảm tải
    frame_truoc_roi = cat_roi(frame_truoc_color, 100, 400, 150, 450)
    frame_truoc_gray = cv2.cvtColor(frame_truoc_roi, cv2.COLOR_BGR2GRAY)
else:
    print("Lỗi: Không thể mở camera.")
    cap.release()
    exit()

print("\n=== BẮT ĐẦU CHẠY PIPELINE (Chỉ phân tích trên vùng ROI) ===")
frame_count = 0

try:
    while cap.isOpened() and frame_count < MAX_FRAMES_TEST:
        time.sleep(1.0 / FPS)
        ret, frame_sau_color = cap.read()
        if not ret: break
        frame_count += 1
        
        # TẦNG 3: Cắt ROI
        frame_sau_roi = cat_roi(frame_sau_color, 100, 400, 150, 450)
        
        # Kiểm tra ảnh mờ (Tùy chọn: có thể áp dụng trên khung màu gốc hoặc ROI)
        if kiem_tra_anh_mo(frame_sau_roi, nguong_mo=100):
            # Nếu quá mờ thì bỏ qua khung hình này
            continue
        
        frame_sau_gray = cv2.cvtColor(frame_sau_roi, cv2.COLOR_BGR2GRAY)
        
        # TẦNG 1 & 2: Phát hiện chuyển động + Lọc nhiễu trên ảnh ROI
        total_clean_area = phat_hien_chuyen_dong_co_loc_nhieu(frame_sau_gray, frame_truoc_gray)
        co_chuyen_dong_thuc_su = (total_clean_area >= NGUONG_CHUYEN_DONG)
        
        # TẦNG 4: Máy trạng thái
        kich_hoat_ai = may_trang_thai.cap_nhat(co_chuyen_dong_thuc_su)
        
        print(f"[Frame {frame_count:03d}] ROI Area: {total_clean_area:05.1f} | Bộ đếm máy trạng thái: {may_trang_thai.bo_dem_chuyen_dong}/5")
        
        # TẦNG 5: Chạy Mô hình AI
        if kich_hoat_ai:
            print("\t=> 🚨 ĐÃ ĐỦ ĐIỀU KIỆN! KÍCH HOẠT TẦNG 5 (AI)")
            if has_model:
                # Đưa toàn bộ khung hình, hoặc chỉ khung ROI vào AI
                phat_hien, diem_so = ai_model.nhan_dien(frame_sau_color)
                if phat_hien:
                    print(f"\t=> 🎯 AI XÁC NHẬN CÓ ĐỐI TƯỢNG HỢP LỆ! (Độ tin cậy: {diem_so:.2f})")
                else:
                    print(f"\t=> ❌ AI KHÔNG TÌM THẤY ĐỐI TƯỢNG! (Độ tin cậy max: {diem_so:.2f})")
        
        # Chuẩn bị cho khung hình tiếp theo
        frame_truoc_gray = frame_sau_gray
        
except KeyboardInterrupt:
    print("\nĐã dừng chạy.")
finally:
    cap.release()
    print("\n=== KẾT THÚC ===")
